In [ ]:
from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
import logging
from datetime import datetime
import os
from logging import FileHandler
import tqdm
from collections import deque

In [ ]:
def run_node(logger, rules, seed=None, events=None):
    if events is not None:
        events = deque(events)
    if seed is None:
        seed = time.time_ns() % (2**32)
        
    np.random.seed(seed)
    logger.info(f"New round with random seed {seed}")
    logger.info("")

    node = BJRound(rules)
    node.start_round(10)

    while node.get_stage() != BJStage.ROUND_OVER:
        logger.info(f"Stage {node.get_stage()}")
        ranks = node.get_possible_next_card_ranks()
        actions = node.get_available_actions()

        if len(ranks) > 0 and len(actions) > 0:
            raise RuntimeError

        if len(ranks) > 0:
            if events is not None:
                rank = events.popleft()
            else:
                rank_idx = np.random.choice(len(ranks))
                rank = ranks[rank_idx]
            card = Card(rank)
            logger.info(f"Pick {rank.value} from [" + " ".join([r.value for r in ranks]) + "]") 
            node.take_card(card)

        if len(actions) > 0:
            if events is not None:
                action = events.popleft()
            else:
                action_idx = np.random.choice(len(actions))
                action = actions[action_idx]
            logger.info(f"Pick {action.value} from [" + " ".join([a.value for a in actions]) + "]")
            node.take_action(action)

        logger.info(node)
        logger.info("")

In [3]:
rules = BJRules()

rules.dealer_hits_soft_17 = True

rules.allow_late_surrender = True
rules.allow_early_surrender_on_all = True
rules.allow_insurance_vs_ace = True
rules.max_splits_allowed = None

# rules.allow_late_surrender = False
# rules.allow_early_surrender_on_all = True
# rules.allow_insurance_vs_ace = True
# rules.max_splits_allowed = 5
# rules.dealer_shows_card_on_surrender = True
# rules.allow_insurance_vs_ace = False
# rules.no_natural_bj_on_split = False
# rules.allow_double_on_soft = False
# rules.allow_action_on_split_aces = False
# rules.allow_double_after_split = False
# rules.dealer_shows_card_on_surrender

print(rules)

BJRules: 
 - dealer_checks_blackjack: True
 - dealer_hits_soft_17: True
 - allow_late_surrender: True
 - allow_early_surrender_on_ten: False
 - allow_early_surrender_on_ace: False
 - allow_early_surrender_on_all: True
 - dealer_shows_card_on_surrender: False
 - allow_insurance_vs_ace: True
 - natural_blackjack_payout: 1.5
 - surrender_payout: 0.5
 - insurance_payout: 2.0
 - max_splits_allowed: None
 - allow_action_on_split_aces: True
 - allow_double_after_split: True
 - allow_double_on_soft: True
 - allow_split_different_tens: True
 - no_natural_bj_on_split: True


In [ ]:
# Ensure logs directory exists
os.makedirs("logs", exist_ok=True)

# Create time-stamped logfile path
log_path = os.path.join(
    "logs", f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
)

# Configure logger
logger = logging.getLogger("BJRoundTest")
logger.setLevel(logging.INFO)

file_handler = FileHandler(log_path)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

print(f"Logging to {log_path}")

Logging to logs/log_20251018_222922.txt


In [ ]:
logger = logging.getLogger("BJRoundTest")
logger.setLevel(logging.INFO)
print(logger.hasHandlers())

True


In [6]:
logger.info(str(rules))
logger.info("")

n_rounds = 400
for i in tqdm.tqdm(range(n_rounds)):
    seed = time.time_ns() % (2**32)
    run_node(logger, rules, seed)

  0%|          | 0/400 [00:00<?, ?it/s]

100%|██████████| 400/400 [00:00<00:00, 1664.03it/s]


In [7]:
import sys


def get_cell_logger(name="cell.logger", level=logging.INFO, stream=sys.stdout):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False
    # Replace handlers so reruns don't stack them
    if logger.handlers:
        logger.handlers.clear()
    h = logging.StreamHandler(stream)
    logger.addHandler(h)
    return logger

In [8]:
seed = 2371240583
standard_logger = get_cell_logger()
standard_logger.setLevel(logging.INFO)

run_node(standard_logger, rules, seed)

New round with random seed 2371240583

Stage BJStage.PLAYER_CARD
Pick T from [A K Q J T 9 8 7 6 5 4 3 2]
Dealer (no cards)
Player T.(10)[$10]

Stage BJStage.PLAYER_CARD
Pick K from [A K Q J T 9 8 7 6 5 4 3 2]
Dealer (no cards)
Player T.K.(20)[$10]

Stage BJStage.DEALER_CARD
Pick 8 from [A K Q J T 9 8 7 6 5 4 3 2]
Dealer 8.X
Player T.K.(20)[$10]

Stage BJStage.PLAYER_OFFERED_EARLY_SURRENDER
Pick DECLINE_EARLY_SURRENDER from [SURRENDER DECLINE_EARLY_SURRENDER]
Dealer 8.X
Player T.K.(20)[$10]

Stage BJStage.PLAYER_ACTION
Pick SPLIT from [STAND HIT DOUBLE SPLIT SURRENDER]
Dealer 8.X
Player T.(10)[$10] | K.(10)[$10]

Stage BJStage.PLAYER_CARD
Pick 7 from [A K Q J T 9 8 7 6 5 4 3 2]
Dealer 8.X
Player T.7.(17)[$10] | K.(10)[$10]

Stage BJStage.PLAYER_CARD
Pick A from [A K Q J T 9 8 7 6 5 4 3 2]
Dealer 8.X
Player T.7.(17)[$10] | K.A.(21/11 - stand)[$10]

Stage BJStage.PLAYER_ACTION
Pick HIT from [STAND HIT DOUBLE]
Dealer 8.X
Player T.7.(17)[$10] | K.A.(21/11 - stand)[$10]

Stage BJStage.PLAYER

In [45]:
scenarios = [
    {
        "name": "split_aces_two_card_21_not_blackjack",
        "goal": "After splitting A,A, draw T on first hand to make a 2-card 21; ensure it scores as 21 (even money), not blackjack.",
        "steps": [
            # Player initial cards
            "A", "A",
            # Dealer upcard (no BJ check stage here)
            "8",
            # Split aces; engine will then ask for one card for each split hand
            "SPLIT",
            # First split hand draw (2-card 21)
            "T",
            # Second split hand draw
            "9",
            # Actions per hand
            # "STAND",   # auto-stand on 21
            "STAND",   # stand on 20
            # Dealer hole + draw to bust so we can verify payout cleanly
            "8", "9"   # dealer: 8.8 (16) -> +9 (25 bust)
        ]
    },
    {
        "name": "max_splits_enforced_one",
        "goal": "With max_splits=1, verify a second split is not offered after the first split creates another pair.",
        "steps": [
            # Player 8,8 vs 6 to encourage split
            "8", "8", "6",
            "SPLIT",
            # Engine deals one card to each split hand; make first hand another 8
            "8",  # first hand now 8,8 again
            "2",  # second hand 8,2
            # At the next PLAYER_ACTION on the first hand, check that 'SPLIT' is NOT in the action list.
            # Then play out normally:
            "HIT", "3", "STAND",    # first hand: 8,8,3 -> 19
            "HIT", "T", "STAND",    # second hand: 8,2,T -> 20
            # Dealer resolves (keep simple)
            "9", "7"  # dealer: 6.9 (15) -> +7 (22 bust)
        ]
    },
    {
        "name": "split_mixed_tens_allowed",
        "goal": "Verify splitting different tens (e.g., K,Q) is allowed and resolves correctly.",
        "steps": [
            "K", "Q", "6",   # Player K,Q vs dealer 6
            "SPLIT",
            # One card to each split hand
            "5",  # first: K,5 (15)
            "4",  # second: Q,4 (14)
            # Actions
            "STAND",                # first hand 15 stands (edge case)
            "DOUBLE", "7",          # second hand doubles: 14+7=21 (one card only)
            # Dealer play (ensure clear compare)
            "9", "7"  # dealer: 6.9 (15) -> +7 (22 bust)
        ]
    },
    {
        "name": "double_after_split_non_aces",
        "goal": "Double after split is allowed on non-ace pairs; verify one-card draw and bet accounting.",
        "steps": [
            "9", "9", "3",
            "SPLIT",
            # One card to each split hand
            "2",  # first: 9,2 (11)
            "2",  # second: 9,2 (11)
            # Actions
            "DOUBLE", "T",         # first: 11+T=21 (stand auto)
            "DOUBLE", "9",         # second: 11+9=20 (stand auto)
            # Dealer play
            "7", "7"               # dealer: 3.7 (10) -> +7 (17 stand)
        ]
    },
    {
        "name": "insurance_then_continue_with_split",
        "goal": "Take insurance vs Ace, dealer has NO blackjack, then split tens and continue.",
        "steps": [
            "T", "T", "A",                  # Player T,T; dealer Ace
            "TAKE_INSURANCE",
            "CONFIRM_NO_BLACKJACK",         # dealer doesn’t have BJ → continue
            "SPLIT",
            # One card to each split hand
            "6",  # first: T,6 (16)
            "9",  # second: T,9 (19)
            # Actions
            "HIT", "2", "STAND",            # first to 18 then stand
            "STAND",
            # Dealer play
            "7", "9"                        # dealer: A.7 (18) -> +9 (27 bust)  [note: if soft logic stands, adjust draws in your harness]
        ]
    },
    {
        "name": "insurance_player_blackjack_push_pays_sidebet",
        "goal": "Player has blackjack vs dealer Ace; take insurance; dealer HAS blackjack → main hand pushes, insurance pays 2:1.",
        "steps": [
            "A", "T", "A",           # Player BJ; dealer Ace
            "TAKE_INSURANCE",
            "CONFIRM_BLACKJACK",     # dealer has BJ
            "T"
            # No further actions; round resolves
        ]
    },
    {
        "name": "refuse_insurance_dealer_blackjack_loses",
        "goal": "Refuse insurance vs Ace when dealer HAS blackjack → lose main bet.",
        "steps": [
            "9", "9", "A",
            "REFUSE_INSURANCE",
            "CONFIRM_BLACKJACK",
            "K"
            # Round resolves as loss
        ]
    },
    {
        "name": "soft_double_allowed",
        "goal": "Double on soft totals is allowed; verify single-card draw and auto-stand after double.",
        "steps": [
            "A", "6", "5",      # A,6 vs 5
            "DOUBLE", "9",      # draw one card → 16/6 + 9 = 15 (if soft converts) /  (engine should auto-stand after double)
            # Dealer play to resolve
            "9", "7"            # dealer: 5.9 (14) -> +7 (21)
        ]
    },
    {
        "name": "dealer_soft17_stands",
        "goal": "Force dealer to soft 17 (A,6) and ensure they stand (dealer_hits_soft_17=False).",
        "steps": [
            "9", "7", "A",      # Player 16 vs dealer Ace (any player line is fine)
            "REFUSE_INSURANCE",
            "CONFIRM_NO_BLACKJACK",
            # Finish player turn quickly
            "STAND",
            # Dealer hole + (no) hit: make dealer A,6 (soft 17)
            "6"
            # Expect dealer to stop here with 17
        ]
    },
    {
        "name": "no_surrender_offered",
        "goal": "With surrender disabled, ensure SURRENDER never appears even vs 10 or Ace upcards.",
        "steps": [
            # Case vs 10
            "9", "7", "T",
            "CONFIRM_NO_BLACKJACK",
            # At PLAYER_ACTION, check action list contains no 'SURRENDER'
            "STAND",
            # Case vs Ace
            "5", "6", "A",
            "REFUSE_INSURANCE",
            # Again, at PLAYER_ACTION, ensure no 'SURRENDER'
            "STAND",
            # Let dealer resolve quickly
            "T"  # dealer hits once (example)
        ]
    },
        {
        "name": "split_aces_hit_multiple_times",
        "goal": "After splitting A,A, verify multiple HITs are allowed (allow_action_on_split_aces=True).",
        "steps": [
            "A", "A", "6",
            "SPLIT",
            "4",            # first hand gets 4  -> A,4
            "9",            # second hand gets 9 -> A,9
            "HIT", "2",     # first hand: A,4,2
            "HIT", "9",     # first hand: A,4,2,9
            "STAND",
            "HIT", "2",     # second hand: A,9,2
            "STAND",
            "7", "8"        # dealer resolves: 6.7(13) +8 -> 21
        ]
    },
    {
        "name": "double_after_split_aces",
        "goal": "After splitting aces, DOUBLE is allowed; verify one-card draw and accounting.",
        "steps": [
            "A", "A", "5",
            "SPLIT",
            "7",            # first hand -> A,7 (soft 18)
            "6",            # second hand -> A,6 (soft 17)
            "DOUBLE", "3",  # first hand doubles to 21 (auto-stand)
            "HIT", "4",     # second hand hits to soft 21
            # "STAND", autostand
            "6", "9"        # dealer: 5.6(11) +9 -> 20
        ]
    },
    {
        "name": "split_tens_two_card_21_not_blackjack",
        "goal": "Split tens; draw Ace on one hand for a 2-card 21 that should NOT count as BJ (no_natural_bj_on_split=True).",
        "steps": [
            "K", "T", "8",
            "SPLIT",
            "A",            # first hand: K,A -> 21 (not BJ)
            "9",            # second hand: T,9 -> 19
            # "STAND", autostand
            "STAND",
            "8", "9"        # dealer: 8.8(16) +9 -> 25 (bust)
        ]
    },
    {
        "name": "soft_double_after_split_on_soft",
        "goal": "After a split, allow DOUBLE on a soft total (allow_double_on_soft=True & allow_double_after_split=True).",
        "steps": [
            "7", "7", "3",
            "SPLIT",
            "A",            # first hand: 7,A (soft 18)
            "3",            # second hand: 7,3 (10)
            "DOUBLE", "2",  # first hand doubles to 20 (auto-stand)
            "DOUBLE", "K",  # second hand doubles to 20 (auto-stand)
            "7", "7"        # dealer: 3.7(10) +7 -> 17
        ]
    },
    {
        "name": "insurance_not_offered_vs_ten",
        "goal": "Vs a Ten upcard, confirm there is no insurance offer (only a BJ check).",
        "steps": [
            "9", "7", "T",
            "CONFIRM_NO_BLACKJACK",  # dealer check only; no insurance stage expected
            "STAND",
            "6", "T"                 # dealer: T.6(16) +T -> 26 (bust)
        ]
    },
    {
        "name": "split_tens_21_push_vs_dealer_21",
        "goal": "After splitting tens, make a 2-card 21 (not BJ) on one hand and have dealer reach 21 to verify it's a push at even money.",
        "steps": [
            "T", "T", "5",      # Player T,T vs dealer 5
            "SPLIT",
            "A",                # First hand: T,A -> 21 (auto-stand, not BJ due to split)
            "9",                # Second hand: T,9 -> 19
            "HIT", "T",         # Second hand hits and busts (19+T=29)
            "9", "7"            # Dealer: 5.9 (14) +7 -> 21 (push vs first hand)
        ]
    }
]

In [ ]:
rules = BJRules()

rules.dealer_checks_blackjack = True
rules.dealer_hits_soft_17 = False
rules.allow_late_surrender = False
rules.allow_early_surrender_on_ten = False
rules.allow_early_surrender_on_ace = False
rules.allow_early_surrender_on_all = False
rules.dealer_shows_card_on_surrender = False
rules.allow_insurance_vs_ace = True
rules.natural_blackjack_payout = 1.5
rules.surrender_payout = 0.5
rules.insurance_payout = 2.0
rules.max_splits_allowed = 1
rules.allow_action_on_split_aces = True
rules.allow_double_after_split = True
rules.allow_double_on_soft = True
rules.allow_split_different_tens = True
rules.no_natural_bj_on_split = True


standard_logger = get_cell_logger()
standard_logger.setLevel(logging.INFO)


print(rules)

for scenario in scenarios[-1:]:
    standard_logger.info(f"Running scenario: {scenario['name']}")
    standard_logger.info(f"Goal: {scenario['goal']}")
    events = []
    for step in scenario["steps"]:
        classes = [Rank, PlayerAction, DealerAction]
        success = False
        for enum_class in classes:
            try:
                events.append(enum_class(step))
                success = True
                break
            except ValueError:
                continue
        if not success:
            raise RuntimeError("conversion failed for " + step)
    print(events)
    run_node(standard_logger, rules, None, events)


BJRules: 
 - dealer_checks_blackjack: True
 - dealer_hits_soft_17: False
 - allow_late_surrender: False
 - allow_early_surrender_on_ten: False
 - allow_early_surrender_on_ace: False
 - allow_early_surrender_on_all: False
 - dealer_shows_card_on_surrender: False
 - allow_insurance_vs_ace: True
 - natural_blackjack_payout: 1.5
 - surrender_payout: 0.5
 - insurance_payout: 2.0
 - max_splits_allowed: 1
 - allow_action_on_split_aces: True
 - allow_double_after_split: True
 - allow_double_on_soft: True
 - allow_split_different_tens: True
 - no_natural_bj_on_split: True
Running scenario: split_tens_21_push_vs_dealer_21
Goal: After splitting tens, make a 2-card 21 (not BJ) on one hand and have dealer reach 21 to verify it's a push at even money.
[<Rank.TEN: 'T'>, <Rank.TEN: 'T'>, <Rank.FIVE: '5'>, <Action.SPLIT: 'SPLIT'>, <Rank.ACE: 'A'>, <Rank.NINE: '9'>, <Action.HIT: 'HIT'>, <Rank.TEN: 'T'>, <Rank.NINE: '9'>, <Rank.SEVEN: '7'>]
New round with random seed 961369065

Stage BJStage.PLAYER_CARD


In [ ]:
a
print(rules)

<Action.SPLIT: 'SPLIT'>